In [ ]:
# =====================================================================
# UET RAG DATA PIPELINE
# =====================================================================

In [ ]:
# =====================================================================
# INSTALL
# =====================================================================
!pip install pymupdf pdfplumber tiktoken -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 86.5 MB/s eta 0:00:00


In [ ]:
# =====================================================================
# IMPORT
# =====================================================================
import fitz
import pdfplumber
import tiktoken
import re
import json
import hashlib
import unicodedata
import os
from google.colab import files

TOKEN_ENCODER = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(TOKEN_ENCODER.encode(text))

In [ ]:
# =====================================================================
# CHUẨN HÓA VĂN BẢN
# =====================================================================
def clean_text_expert(text):
    if not text:
        return ""

    text = unicodedata.normalize("NFC", text)

    # XÓA RÁC HÀNH CHÍNH & MỤC LỤC CHẤM CHẤM
    patterns = [
        r'CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM',
        r'Độc lập\s*[-–]?\s*Tự do\s*[-–]?\s*Hạnh phúc',
        r'ĐẠI HỌC QUỐC GIA HÀ NỘI',
        r'TRƯỜNG ĐẠI HỌC CÔNG NGHỆ',
        r'.*?\.{3,}.*?\d+',
    ]
    for p in patterns:
        text = re.sub(p, '', text, flags=re.IGNORECASE | re.MULTILINE)

    text = re.sub(r'\s*\n\s*', ' ', text)

    text = re.sub(r'(?<=[a-zA-ZÀ-ỹ])\s+\d+\s+(?=[a-zA-ZÀ-ỹ])', ' ', text)

    text = re.sub(r'[ \t]+', ' ', text)

    return text.strip()

def is_quality_chunk(text):
    clean_chars = re.sub(r'\s+', '', text)
    if not clean_chars:
        return False
    alpha_chars = len(re.findall(r'[A-Za-zÀ-ỹ0-9]', clean_chars))
    return (alpha_chars / len(clean_chars)) >= 0.55 and count_tokens(text) > 35

def extract_pdf_text(pdf_path, skip_pages=None) -> str:
    doc = fitz.open(pdf_path)
    pages = []
    for page_num in range(len(doc)):
        if skip_pages and page_num in skip_pages:
            continue
        pages.append(doc[page_num].get_text())
    doc.close()
    return "\n".join(pages)

In [ ]:
# =====================================================================
# BỘ TÁCH PHÂN ĐOẠN VĂN XUÔI - MÁY TRẠNG THÁI TUẦN TỰ KHÉP KÍN
# =====================================================================
def segment_and_chunk_document(raw_text, source_tag, is_quy_che=False, target_chunk_tokens=420):
    lines = [l.strip() for l in raw_text.split('\n')]

    HEADERS_MAP = {
        "QUY_CHE_DAO_TAO": [
            "QUY ĐỊNH CHUNG",
            "TỔ CHỨC ĐÀO TẠO",
            "KIỂM TRA VÀ THI HỌC PHẦN",
            "XẤT VÀ CÔNG NHẬN TỐT NGHIỆP"
        ],
        "SO_TAY_HOC_VU": [
            "ĐĂNG KÝ HỌC PHẦN",
            "ĐIỀU KIỆN ĐỂ MIỄN HỌC PHẦN TIẾNG ANH",
            "DANH SÁCH CÁC CƠ SỞ CẤP CHỨNG CHỈ NGOẠI NGỮ",
            "BẢNG THAM CHIẾU QUY ĐỔI MỘT SỐ CHỨNG CHỈ NGOẠI NGỮ",
            "MỘT SỐ ĐIỂM CẦN LƯU Ý TRONG QUY CHẾ ĐÀO TẠO"
        ],
        "TUYEN_SINH_2026": [
            "PHƯƠNG THỨC XÉT TUYỂN",
            "THÔNG TIN TUYỂN SINH",
            "CHỈ TIÊU TUYỂN SINH",
            "TUYỂN SINH ĐÀO TẠO ĐẠI HỌC",
            "THÔNG TIN CHUNG"
        ]
    }

    file_specific_headers = HEADERS_MAP.get(source_tag, [])

    segments = []
    current_large = ""
    current_sub = ""
    current_buffer = []

    def flush_segment():
        if current_buffer:
            content_block = "\n".join(current_buffer).strip()
            if content_block:
                if current_large and current_sub:
                    t_val = f"{current_large} {current_sub}"
                else:
                    t_val = current_large if current_large else current_sub

                pure_txt = content_block
                if current_sub and pure_txt.startswith(current_sub):
                    pure_txt = pure_txt[len(current_sub):].strip()
                elif current_large and pure_txt.startswith(current_large):
                    pure_txt = pure_txt[len(current_large):].strip()

                segments.append({
                    "title": re.sub(r'\s+', ' ', t_val).strip(". "),
                    "raw_text": pure_txt
                })
            current_buffer.clear()

    for line in lines:
        if not line:
            continue
        line_norm = unicodedata.normalize("NFC", line).strip()

        if re.match(r'^\d+\s+[A-Z]{2,}\d*\s+\d+', line_norm) or "SAT Alevel" in line_norm:
            continue

        matched_large = False
        for v_header in file_specific_headers:
            if v_header in line_norm.upper() or line_norm.upper().startswith(v_header):
                flush_segment()
                current_large = v_header
                current_sub = ""
                matched_large = True
                break
        if matched_large:
            continue

        matched_sub = False
        if is_quy_che:
            chuong_m = re.match(r'^(Chương\s+[IVXLCDM]+.*)', line_norm, re.IGNORECASE)
            dieu_m = re.match(r'^(Điều\s+\d+.*)', line_norm, re.IGNORECASE)
            if chuong_m:
                flush_segment()
                current_large = chuong_m.group(1).strip()
                current_sub = ""
                continue
            if dieu_m:
                flush_segment()
                current_sub = dieu_m.group(1).strip()
                matched_sub = True
        else:
            if source_tag == "SO_TAY_HOC_VU":
                sub_m = re.match(r'^(\d+\.\s+.*)', line_norm)
            else:
                if re.match(r'^\d+\.\d{2}\b', line_norm) or re.match(r'^\d+,\d{2}\b', line_norm):
                    sub_m = None
                else:
                    sub_m = re.match(r'^(\d+\.\d+\.?\s*.*)', line_norm)

                if not sub_m:
                    sub_m = re.match(r'^(\d+\.\s+.*)', line_norm)
                if not sub_m:
                    sub_m = re.match(r'^([IVXLCDM]+\.\s*.*)', line_norm)

            if sub_m:
                flush_segment()
                current_sub = sub_m.group(1).strip()
                matched_sub = True

        if matched_sub:
            continue

        current_buffer.append(line)

    flush_segment()

    final_chunks = []
    chunk_index = 0

    for seg in segments:
        clean_text = clean_text_expert(seg["raw_text"])
        clean_text = re.sub(r'^[.,;:\s\-+*•]+', '', clean_text).strip()

        if not clean_text or "Chỉ tiêu Nhập học SAT" in clean_text or "Alevel /ACT ĐGNL" in clean_text:
            continue

        seg_tokens = count_tokens(clean_text)

        if seg_tokens <= 550:
            if is_quality_chunk(clean_text):
                final_chunks.append({
                    "chunk_id": f"{source_tag}_REG_{chunk_index:03d}",
                    "title": seg["title"],
                    "text": clean_text
                })
                chunk_index += 1
        else:
            sentences = re.split(r'(?<=[.])\s+', clean_text)
            sentences = [s.strip() for s in sentences if s.strip()]

            i = 0
            while i < len(sentences):
                chunk_sentences = []
                current_tokens = 0

                while i < len(sentences):
                    sentence = sentences[i]
                    s_tokens = count_tokens(sentence + " ")

                    if len(chunk_sentences) > 0:
                        if is_quy_che and re.match(r'^(Điều\s+\d+)', sentence, re.IGNORECASE):
                            break
                        if source_tag == "SO_TAY_HOC_VU" and re.match(r'^(\d+\.)', sentence):
                            break
                        if source_tag == "TUYEN_SINH_2026" and (re.match(r'^(\d+\.\d+)', sentence) or re.match(r'^([IVXLCDM]+\.)', sentence)):
                            if re.match(r'^\d+\.\d{2}\b', sentence):
                                pass
                            else:
                                break

                    if current_tokens + s_tokens <= target_chunk_tokens:
                        chunk_sentences.append(sentence)
                        current_tokens += s_tokens
                        i += 1
                    else:
                        break

                if chunk_sentences:
                    sub_txt = " ".join(chunk_sentences).strip()
                    sub_txt = re.sub(r'^[.,;:\s\-+*•]+', '', sub_txt).strip()
                    if is_quality_chunk(sub_txt):
                        final_chunks.append({
                            "chunk_id": f"{source_tag}_REG_{chunk_index:03d}",
                            "title": seg["title"],
                            "text": sub_txt
                        })
                        chunk_index += 1
                else:
                    i += 1

    return final_chunks


In [ ]:
# =====================================================================
# CÀO VÀ FLAT TẤT CẢ CÁC BẢNG TRONG ĐỀ ÁN
# =====================================================================
def extract_all_admission_tables(pdf_path, source_tag, skip_pages=None):
    """Hàm thông minh tự động phẳng hóa mọi định dạng bảng biểu xuất hiện trong ảnh."""
    chunks = []
    chunk_index = 0
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            if skip_pages and page_num in skip_pages:
                continue
            tables = page.extract_tables()
            if not tables:
                continue

            for table in tables:
                if not table or len(table) < 2:
                    continue
                header_str = " ".join([str(cell) for cell in table[0] if cell]).upper()

                # -------------------------------------------------------------
                # LOẠI 1: BẢNG CHỨNG CHỈ TIẾNG ANH CHUẨN ĐẦU RA (6 CỘT - HÌNH 1)
                # -------------------------------------------------------------
                if "KNLNNVN" in header_str or "APTIS ESOL" in header_str:
                    for row in table[1:]:
                        if not row or not row[0] or "Bậc" not in str(row[0]): continue
                        cells = [str(c).replace("\n", " ").strip() for c in row]
                        flat_text = (
                            f"Bảng tham chiếu kết quả bài thi tiếng Anh với chuẩn cần đạt của ĐHQGHN: "
                            f"Đối với Khung năng lực ngoại ngữ Việt Nam {cells[0]}, mức điểm tối thiểu cần đạt được của các chứng chỉ tương đương bao gồm: "
                            f"Chứng chỉ IELTS cần đạt mức {cells[1]}; Chứng chỉ TOEFL iBT đạt {cells[2]}; Chứng chỉ Aptis ESOL đạt {cells[3]}; "
                            f"Chứng chỉ Cambridge Exam đạt {cells[4]}; Bài thi đánh giá năng lực tiếng Anh theo định dạng VSTEP đạt mức {cells[5]}."
                        )
                        chunks.append({
                            "chunk_id": f"{source_tag}_TABLE_EN_{chunk_index:03d}",
                            "title": "BẢNG THAM CHIẾU KẾT QUẢ CÁC BÀI THI TIẾNG ANH VỚI CÁC CHUẨN CẦN ĐẠT CỦA ĐHQGHN",
                            "text": clean_text_expert(flat_text)
                        })
                        chunk_index += 1

                # -------------------------------------------------------------
                # LOẠI 2: BẢNG HỌC PHÍ NĂM HỌC 2026-2027 (4 CỘT - HÌNH 2)
                # -------------------------------------------------------------
                elif "HỌC PHÍ NĂM HỌC" in header_str or (len(table[0]) == 4 and "MÃ TUYỂN SINH" in header_str):
                    common_fee = "44,000,000"
                    for row in table[1:]:
                        if len(row) >= 4 and row[3] and "44" in str(row[3]):
                            common_fee = str(row[3]).replace("\n", " ").strip()
                            break
                    for row in table[1:]:
                        if not row or not row[0] or not str(row[0]).strip().isdigit(): continue
                        cells = [str(c).replace("\n", " ").strip() for c in row]
                        flat_text = (
                            f"Thông tin học phí Trường Đại học Công nghệ - ĐHQGHN: "
                            f"Ngành hoặc chương trình đào tạo {cells[2]}, có mã tuyển sinh là {cells[1]}. "
                            f"Mức học phí dự kiến áp dụng cho năm học 2026-2027 là {common_fee} đồng/năm học (dự kiến năm thứ nhất học tại Hòa Lạc)."
                        )
                        chunks.append({
                            "chunk_id": f"{source_tag}_TABLE_TUITION_{chunk_index:03d}",
                            "title": f"Mức học phí năm học 2026-2027 ngành {cells[2]}",
                            "text": clean_text_expert(flat_text)
                        })
                        chunk_index += 1

                # -------------------------------------------------------------
                # LOẠI 3: BẢNG MÔ TẢ PHƯƠNG THỨC XÉT TUYỂN (3 CỘT - HÌNH 5)
                # -------------------------------------------------------------
                elif "PHƯƠNG THỨC" in header_str and "CHỈ TIÊU" in header_str and len(table[0]) == 3:
                    for row in table[1:]:
                        if not row or not row[0]: continue
                        cells = [str(c).replace("\n", " ").strip() for c in row]
                        flat_text = f"Mô tả phương thức tuyển sinh năm 2026: Phương thức {cells[1]} chiếm chỉ tiêu tỉ lệ là {cells[2]}."
                        chunks.append({
                            "chunk_id": f"{source_tag}_TABLE_METHOD_{chunk_index:03d}",
                            "title": "Mô tả cơ cấu tỷ lệ phương thức tuyển sinh năm 2026",
                            "text": clean_text_expert(flat_text)
                        })
                        chunk_index += 1

                # -------------------------------------------------------------
                # LOẠI 4: BẢNG DANH MỤC MÔN THI ĐẠT GIẢI XÉT TUYỂN THẲNG (HÌNH 6 & HÌNH 7)
                # -------------------------------------------------------------
                elif "MÔN THI ĐẠT GIẢI" in header_str or "NGÀNH XÉT TUYỂN THẲNG" in header_str:
                    for row in table[1:]:
                        if not row or not row[0] or not str(row[0]).strip().isdigit(): continue
                        cells = [str(c).replace("\n", " ").strip() for c in row]
                        flat_text = f"Quy định tuyển thẳng diện học sinh giỏi: Thí sinh đạt giải môn thi {cells[1]} sẽ được đăng ký xét tuyển thẳng hoặc cộng điểm vào: {cells[2]}."
                        chunks.append({
                            "chunk_id": f"{source_tag}_TABLE_PRIZE_{chunk_index:03d}",
                            "title": f"Danh mục môn thi học sinh giỏi quốc gia tuyển thẳng diện môn {cells[1]}",
                            "text": clean_text_expert(flat_text)
                        })
                        chunk_index += 1

                # -------------------------------------------------------------
                # LOẠI 5: BẢNG NGƯỠNG ĐẢM BẢO CHẤT LƯỢNG ĐẦU VÀO ĐBCL (HÌNH 8)
                # -------------------------------------------------------------
                elif "NGƯỠNG ĐBCL" in header_str or ("MÃ XÉT TUYỂN" in header_str and "CHỈ TIÊU" in header_str and len(table[0]) == 5):
                    common_dbcl = "24"
                    for row in table[1:]:
                        if len(row) >= 5 and row[4] and str(row[4]).strip().isdigit():
                            common_dbcl = str(row[4]).strip()
                            break
                    for row in table[1:]:
                        if not row or not row[0] or not str(row[0]).strip().isdigit(): continue
                        cells = [str(c).replace("\n", " ").strip() for c in row]
                        flat_text = f"Thông tin Ngưỡng đảm bảo chất lượng đầu vào (Điểm sàn): Ngành {cells[1]} (Mã xét tuyển: {cells[2]}), có chỉ tiêu diện này là {cells[3]} thí sinh. Ngưỡng điểm đảm bảo chất lượng đầu vào đạt {common_dbcl} điểm."
                        chunks.append({
                            "chunk_id": f"{source_tag}_TABLE_DBCL_{chunk_index:03d}",
                            "title": f"Ngưỡng đảm bảo chất lượng đầu vào điểm sàn ngành {cells[1]}",
                            "text": clean_text_expert(flat_text)
                        })
                        chunk_index += 1

                # -------------------------------------------------------------
                # LOẠI 6: SỬA ĐỔI TOÀN DIỆN MA TRẬN ĐIỂM CHUẨN 14 CỘT (HÌNH 3)
                # -------------------------------------------------------------
                elif "THÔNG TIN VỀ TUYỂN SINH CỦA 2 NĂM" in header_str or len(table[0]) >= 13 or "ĐGNL" in header_str:
                    for row in table:
                        if not row or not row[0]: continue
                        stt_clean = str(row[0]).strip()
                        if not stt_clean.isdigit(): continue

                        cells = [str(c).replace("\n", " ").strip() if (c and str(c).strip() != '-') else "Không có dữ liệu" for c in row]

                        if len(cells) >= 14:
                            cells = cells[:14]
                        else:
                            continue

                        ma_xet_tuyen = cells[1]
                        ma_nganh = cells[2]
                        ten_nganh = cells[3]

                        # Dữ liệu năm 2024
                        chi_tieu_24 = cells[4]
                        nhap_hoc_24 = cells[5]
                        sat_24 = cells[6]
                        alevel_24 = cells[7] if cells[7] != "Không có dữ liệu" else "Không xét"
                        dgnl_24 = cells[8]
                        ielts_24 = cells[9]
                        thpt_24 = cells[10]

                        # Dữ liệu năm 2025
                        chi_tieu_25 = cells[11]
                        nhap_hoc_25 = cells[12]
                        diem_chuan_25 = cells[13]

                        natural_text = (
                            f"Thông tin tuyển sinh và điểm chuẩn ngành {ten_nganh} (Mã xét tuyển: {ma_xet_tuyen}, Mã ngành: {ma_nganh}) của Trường Đại học Công nghệ. "
                            f"Dữ liệu tuyển sinh năm 2024 bao gồm: Chỉ tiêu tuyển sinh là {chi_tieu_24}; Số lượng sinh viên thực tế nhập học là {nhap_hoc_24}. "
                            f"Điểm trúng tuyển tối thiểu của năm 2024 theo các phương thức xét tuyển cụ thể là: Phương thức điểm SAT đạt {sat_24}, "
                            f"Phương thức Alevel/ACT đạt {alevel_24}, Phương thức thi Đánh giá năng lực (ĐGNL ĐHQGHN) đạt {dgnl_24} điểm, "
                            f"Phương thức chứng chỉ quốc tế IELTS đạt {ielts_24}, và Phương thức điểm thi tốt nghiệp THPT đạt {thpt_24} điểm. "
                            f"Dữ liệu tuyển sinh năm 2025 bao gồm: Chỉ tiêu tuyển sinh là {chi_tieu_25}; Số lượng sinh viên nhập học là {nhap_hoc_25}; "
                            f"Điểm chuẩn trúng tuyển chính thức vào ngành này năm 2025 đạt mức {diem_chuan_25} điểm."
                        )
                        chunks.append({
                            "chunk_id": f"{source_tag}_TABLE_AD_{chunk_index:03d}",
                            "title": f"Thông tin về chỉ tiêu và điểm chuẩn trúng tuyển 2 năm gần nhất ngành {ten_nganh}",
                            "text": clean_text_expert(natural_text)
                        })
                        chunk_index += 1
    return chunks

In [ ]:
# =====================================================================
# KHỬ TRÙNG LẶP DỰ THỪA (DEDUPLICATE)
# =====================================================================
def deduplicate_chunks(chunks):
    seen = set()
    unique = []
    for chunk in chunks:
        normalized = re.sub(r'[^a-zA-ZÀ-ỹ0-9]', '', chunk["text"].lower())
        h = hashlib.md5(normalized.encode("utf-8")).hexdigest()
        if h not in seen:
            seen.add(h)
            unique.append(chunk)
    return unique

In [ ]:
# =====================================================================
# RUN MAIN PIPELINE
# =====================================================================
def run_pipeline():
    print("=" * 70)
    print("UET RAG DATA PIPELINE - FULL SYSTEM SYNCHRONIZATION")
    print("=" * 70)

    path_so_tay = "/content/SỔ-TAY-HỌC-VỤ-KỲ-I-NĂM-2023-2024.pdf"
    path_quy_che = "/content/Final_QC-dH-_2014_Ban-hanh-25-12-2014.pdf"
    path_tuyen_sing = "/content/Thong-tin-tuyen-sinh-DHCQ-nam-2026-QHI.pdf"

    if not os.path.exists(path_so_tay) or not os.path.exists(path_quy_che) or not os.path.exists(path_tuyen_sing):
        print("[ERROR] Thiếu tệp tin tuyển sinh trong thư mục /content/. Vui lòng tải đầy đủ file lên.")
        return

    all_chunks = []

    # 1. SỔ TAY HỌC VỤ
    print("\n[1/3] Processing Sổ tay học vụ...")
    skip_so_tay = [0, 1, 2, 32, 34, 35, 36, 37] + list(range(6, 25))
    text_so_tay = extract_pdf_text(path_so_tay, skip_pages=skip_so_tay)
    all_chunks.extend(segment_and_chunk_document(text_so_tay, "SO_TAY_HOC_VU", is_quy_che=False))
    all_chunks.extend(extract_all_admission_tables(path_so_tay, "SO_TAY_HOC_VU", skip_pages=skip_so_tay))

    # 2. QUY CHẾ ĐÀO TẠO
    print("[2/3] Processing Quy chế đào tạo...")
    text_quy_che = extract_pdf_text(path_quy_che)
    all_chunks.extend(segment_and_chunk_document(text_quy_che, "QUY_CHE_DAO_TAO", is_quy_che=True))

    # 3. TUYỂN SINH
    print("[3/3] Processing Tuyển sinh...")
    text_tuyen_sinh = extract_pdf_text(path_tuyen_sing)
    all_chunks.extend(segment_and_chunk_document(text_tuyen_sinh, "TUYEN_SINH_2026", is_quy_che=False))
    all_chunks.extend(extract_all_admission_tables(path_tuyen_sing, "TUYEN_SINH_2026"))

    final_chunks = deduplicate_chunks(all_chunks)
    output_file = "/content/uet_rag_chunks_dataset.json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(final_chunks, f, ensure_ascii=False, indent=2)

    print(f"\n[THÀNH CÔNG] Đã cấu trúc lại ma trận bảng điểm chuẩn.")
    print(f"Tệp JSON siêu sạch sẵn sàng tải về: {output_file}")
    files.download(output_file)

In [ ]:
run_pipeline()

UET RAG DATA PIPELINE - FULL SYSTEM SYNCHRONIZATION

[1/3] Processing Sổ tay học vụ...
[2/3] Processing Quy chế đào tạo...
[3/3] Processing Tuyển sinh...

[THÀNH CÔNG] Đã cấu trúc lại ma trận bảng điểm chuẩn.
Tệp JSON siêu sạch sẵn sàng tải về: /content/uet_rag_chunks_dataset.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>